# 03 · Feature engineering

**Input:** `data/interim/rdu_temperature_clean_2015-01-01_to_2026-09-16.csv` (from `02`) and `data/raw/ecmwf_ifs_runs_2024-03-14_to_2026-09-16.csv` (from `01`).
**Output:** `data/processed/model_samples_2024-03-14_to_2026-09-16.csv.gz`, the model matrix X (and target y) for the linear regression in `04`, plus the same table uncompressed (`.csv`) for viewing.

**One row = one forecast hour.** For every day *d* from Mar 14, 2024 to Sep 16, 2026, a forecast is issued at 11pm local (the last observed report, 11:51pm) using the ECMWF 12z run of day *d*, and covers the next 336 hours (12am day *d*+1 … 11pm day *d*+14). The last block (Sep 16, 2026) is the real forecast for Sep 17–30; its target is unknown.

**Rule:** every feature uses only information available at 11:51pm on day *d*.

## 1. Settings and hourly grid

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    folder for folder in [current_dir, *current_dir.parents]
    if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file()
)
CLEAN_FILE = PROJECT_ROOT / "data" / "interim" / "rdu_temperature_clean_2015-01-01_to_2026-09-16.csv"
ECMWF_FILE = PROJECT_ROOT / "data" / "raw" / "ecmwf_ifs_runs_2024-03-14_to_2026-09-16.csv"
SAMPLES_FILE = PROJECT_ROOT / "data" / "processed" / "model_samples_2024-03-14_to_2026-09-16.csv.gz"  # read by 04 and 05
SAMPLES_VIEW_FILE = SAMPLES_FILE.with_suffix("")  # same table as a plain .csv, for viewing

NY = "America/New_York"
CUTOFF = pd.Timestamp("2026-09-17", tz=NY)        # 12am Sep 17: first forecast hour
FORECAST_END = pd.Timestamp("2026-10-01", tz=NY)  # exclusive: last forecast hour is 11pm Sep 30
HORIZON = 336                                     # forecast hours per run
FIRST_RUN_DAY, LAST_RUN_DAY = "2024-03-14", "2026-09-16"
NWP_OFFSET = pd.Timedelta(hours=1)                # report labelled hh (taken at hh:51) <-> forecast valid at hh+1:00

clean = pd.read_csv(CLEAN_FILE, index_col="hour_utc", parse_dates=["hour_utc"])
# Extend the hourly grid by the 336 forecast hours: their temperature is unknown, but their calendar terms are not
grid = pd.date_range(clean.index[0], FORECAST_END.tz_convert("UTC"), freq="h", inclusive="left", name="hour_utc")
hourly = pd.DataFrame({"temperature": clean["temperature"].reindex(grid)}, index=grid)
local = grid.tz_convert(NY)
hourly["year"], hourly["doy"], hourly["hour"] = local.year, local.dayofyear, local.hour
assert hourly.loc[hourly.index >= CUTOFF, "temperature"].isna().all()
print(f"Hourly grid: {len(hourly):,} hours, {local[0]:%Y-%m-%d %H:%M} -> {local[-1]:%Y-%m-%d %H:%M} Eastern "
      f"({(hourly.index >= CUTOFF).sum()} forecast hours with unknown temperature)")

Hourly grid: 102,983 hours, 2015-01-01 00:00 -> 2026-09-30 23:00 Eastern (336 forecast hours with unknown temperature)


## 2. Station features (NOAA)

| Feature | Meaning |
|---|---|
| `hist_avg` | historical average: mean temperature at the same day of year and hour in **earlier years**, smoothed over ±3 days. It is both a baseline and the base of the model. |
| `climatology` | smooth seasonal + daily curve (linear regression on calendar sin/cos terms) fitted on earlier years; only used to define the anomaly |
| `anom_now`, `anom_24h` | how much warmer than normal the station is at the forecast start, and on average over the last 24 hours |

For every year Y, `hist_avg` and `climatology` are computed from the years before Y only.

In [2]:
# Cyclical calendar terms (position in the day and in the year)
day_fraction = hourly["hour"] / 24
year_fraction = (hourly["doy"] - 1 + day_fraction) / np.where(local.is_leap_year, 366, 365)
for k in (1, 2):
    hourly[f"hour_sin{k}"], hourly[f"hour_cos{k}"] = np.sin(2 * np.pi * k * day_fraction), np.cos(2 * np.pi * k * day_fraction)
    hourly[f"doy_sin{k}"], hourly[f"doy_cos{k}"] = np.sin(2 * np.pi * k * year_fraction), np.cos(2 * np.pi * k * year_fraction)
for season in ("doy_sin1", "doy_cos1"):
    for daily in ("hour_sin1", "hour_cos1"):
        hourly[f"{season}_x_{daily}"] = hourly[season] * hourly[daily]
CALENDAR = [c for c in hourly.columns if c.startswith(("hour_sin", "hour_cos", "doy_"))]

# For each year Y, only earlier years are used (so nothing is computed from the future)
observed = hourly["temperature"].notna()
hourly["hist_avg"] = np.nan      # historical average: same day of year and hour, earlier years, smoothed +-3 days
hourly["climatology"] = np.nan   # smooth seasonal + daily curve fitted on earlier years (used for the anomaly)
for year in range(hourly["year"].min() + 1, hourly["year"].max() + 1):
    past, rows = observed & hourly["year"].lt(year), hourly["year"].eq(year)
    means = hourly[past].groupby(["doy", "hour"])["temperature"].mean().unstack().reindex(range(1, 367))
    smooth = pd.concat([means.iloc[-3:], means, means.iloc[:3]]).rolling(7, center=True, min_periods=1).mean().iloc[3:-3]
    hourly.loc[rows, "hist_avg"] = smooth.to_numpy()[hourly.loc[rows, "doy"] - 1, hourly.loc[rows, "hour"]]
    model = LinearRegression().fit(hourly.loc[past, CALENDAR], hourly.loc[past, "temperature"])
    hourly.loc[rows, "climatology"] = model.predict(hourly.loc[rows, CALENDAR])

# Recent station anomaly (how much warmer than normal it is), from past hours only
anomaly = hourly["temperature"] - hourly["climatology"]
hourly["anom_now"] = anomaly.ffill(limit=3)                       # bridge gaps of up to 3 hours
hourly["anom_24h"] = anomaly.rolling(24, min_periods=18).mean()   # trailing windows: hour t and earlier
hourly["anom_7d"] = anomaly.rolling(168, min_periods=126).mean()  # only used to fill gaps in the two above
display(hourly.loc[hourly.index < CUTOFF, ["temperature", "hist_avg", "climatology", "anom_now", "anom_24h"]].describe().round(2))

,temperature,hist_avg,climatology,anom_now,anom_24h
count,102462.00,93887.00,93887.00,93795.00,93747.00
mean,16.91,16.68,16.69,0.28,0.28
std,9.34,8.23,8.07,4.84,4.32
min,-15.60,-8.50,1.51,-19.76,-14.89
25%,10.00,9.73,9.82,-2.86,-2.50
50%,18.30,17.14,16.87,0.35,0.25
75%,23.90,23.27,23.17,3.23,2.85
max,40.00,35.31,32.52,18.73,15.32


## 3. ECMWF features

| Feature | Meaning |
|---|---|
| `ecmwf_12z`, `ecmwf_00z`, `ecmwf_prev12z` | forecast temperature for the target hour from the 12z and 00z runs of day *d* and the 12z run of day *d*−1, all published before the forecast start |
| `ecmwf_lagged_mean` | mean of the three runs (a time-lagged ensemble), smoother at long leads |
| `nwp_recent_error` | observed − forecast over the last 6 observed hours (6–11pm on day *d*): how wrong this run already is |
| `nwp_bias_14d/30d/60d` | mean observed − forecast over the previous 14 / 30 / 60 days (leads 17–40 h of the runs from day *d*−2 backwards, all verified before the forecast start): the recent systematic bias |

In [4]:
ecmwf = pd.read_csv(ECMWF_FILE, usecols=["run_utc", "valid_utc", "lead_hours", "temperature_2m"],
                    parse_dates=["run_utc", "valid_utc"])
forecast = ecmwf.set_index(["run_utc", "valid_utc"])["temperature_2m"]
lookup = lambda runs, valid: forecast.reindex(pd.MultiIndex.from_arrays([runs, valid])).to_numpy()

run_days = pd.date_range(FIRST_RUN_DAY, LAST_RUN_DAY, freq="D")
runs = run_days.tz_localize("UTC") + pd.Timedelta(hours=12)                       # the 12z run of each day
origins = pd.DatetimeIndex([pd.Timestamp(d.year, d.month, d.day, 23, tz=NY) for d in run_days]).tz_convert("UTC")
origin_pos = hourly.index.get_indexer(origins)                                    # 11pm local: last observed hour
temperature = hourly["temperature"].to_numpy()

# Recent ECMWF error: observed minus forecast over the last 6 observed hours (6pm-11pm local on the run day)
last_six = pd.DataFrame({back: temperature[origin_pos - back] - lookup(runs, hourly.index[origin_pos - back] + NWP_OFFSET)
                         for back in range(6)}, index=runs)
recent_error = last_six.mean(axis=1)  # NaN only if none of the six hours can be compared

# Rolling ECMWF bias: mean(observed - forecast) at leads 17-40 h of the runs 2 to N days before (all verified before the origin)
verify = ecmwf[ecmwf["run_utc"].dt.hour.eq(12) & ecmwf["lead_hours"].between(17, 40)]
error = hourly["temperature"].reindex(verify["valid_utc"] - NWP_OFFSET).to_numpy() - verify["temperature_2m"].to_numpy()
daily_error = pd.Series(error).groupby(verify["run_utc"].to_numpy()).mean()
daily_error = daily_error.reindex(pd.date_range(daily_error.index.min(), daily_error.index.max(), freq="D"))
bias = {w: daily_error.shift(2).rolling(w, min_periods=w // 2).mean() for w in (14, 30, 60)}
print(f"12z runs: {ecmwf.loc[ecmwf['run_utc'].dt.hour.eq(12), 'run_utc'].nunique()} | recent error available for "
      f"{recent_error.notna().sum()} of {len(runs)} run days | 30-day bias available for {bias[30].notna().sum()}")

12z runs: 915 | recent error available for 909 of 917 run days | 30-day bias available for 901


## 4. Assemble X

- `lead_day`: forecast day 1–14 (`04` fits one regression per lead day).
- **Model inputs (12):** `nwp_lagged_anomaly` and `nwp_anomaly` (ECMWF minus `hist_avg`), `nwp_recent_error`, the three biases, `anom_now`, `anom_24h`, and the target hour as sin/cos.
- **Target:** `temperature`; the model predicts `temperature − hist_avg`.
- Kept for comparison: `persistence` (same clock hour on the last observed day), `hist_avg` and the raw ECMWF columns.
- A few gaps in the station anomaly and the recent error are filled with information from the same moment (the 24-hour mean, the 30-day bias). Rows without an ECMWF forecast (runs missing from the archive, leads beyond the run length) keep `NaN`.

In [5]:
step = np.tile(np.arange(1, HORIZON + 1), len(runs))      # hours after the 11pm origin (1 = 12am the next day)
o = np.repeat(origin_pos, HORIZON)
t = o + step
run = np.repeat(runs, HORIZON)
target_utc = hourly.index[t]
valid = target_utc + NWP_OFFSET
col = lambda name, pos: hourly[name].to_numpy()[pos]

samples = pd.DataFrame({
    "run_day": np.repeat(run_days, HORIZON),
    "target_utc": target_utc,
    "target_local": target_utc.tz_convert(NY).strftime("%Y-%m-%d %H:%M"),
    "lead_day": (step - 1) // 24 + 1,
    "temperature": col("temperature", t),                                # y (unknown for the final forecast)
    "hist_avg": col("hist_avg", t),
    "persistence": col("temperature", t - 24 * np.ceil(step / 24).astype(int)),  # same clock hour, last observed day
    "ecmwf_12z": lookup(run, valid),                                     # run of the same day (12z)
    "ecmwf_00z": lookup(run - pd.Timedelta(hours=12), valid),            # 00z run of the same day
    "ecmwf_prev12z": lookup(run - pd.Timedelta(hours=24), valid),        # 12z run of the day before
    "nwp_recent_error": recent_error.reindex(run).to_numpy(),
    **{f"nwp_bias_{w}d": bias[w].reindex(run).to_numpy() for w in (14, 30, 60)},
    **{c: col(c, o) for c in ["anom_now", "anom_24h", "anom_7d"]},
    **{c: col(c, t) for c in ["hour_sin1", "hour_cos1", "hour_sin2", "hour_cos2"]},
})
samples["ecmwf_lagged_mean"] = samples[["ecmwf_12z", "ecmwf_00z", "ecmwf_prev12z"]].mean(axis=1).where(samples["ecmwf_12z"].notna())

# Fill a few gaps with information from the same moment (no look-ahead), then build the model inputs
samples["anom_now"] = samples["anom_now"].fillna(samples["anom_24h"]).fillna(samples["anom_7d"]).fillna(0.0)
samples["anom_24h"] = samples["anom_24h"].fillna(samples["anom_now"])
samples["nwp_recent_error"] = samples["nwp_recent_error"].fillna(samples["nwp_bias_30d"]).fillna(0.0)
for w in (14, 30, 60):
    samples[f"nwp_bias_{w}d"] = samples[f"nwp_bias_{w}d"].fillna(0.0)
samples["nwp_lagged_anomaly"] = samples["ecmwf_lagged_mean"] - samples["hist_avg"]
samples["nwp_anomaly"] = samples["ecmwf_12z"] - samples["hist_avg"]
samples = samples.drop(columns="anom_7d")

FEATURES = ["nwp_lagged_anomaly", "nwp_anomaly", "nwp_recent_error", "nwp_bias_14d", "nwp_bias_30d", "nwp_bias_60d",
            "anom_now", "anom_24h", "hour_sin1", "hour_cos1", "hour_sin2", "hour_cos2"]
print(f"X: {len(samples):,} rows ({len(runs)} run days x {HORIZON} hours), {len(FEATURES)} features; "
      f"rows with a known target: {samples['temperature'].notna().sum():,}")
display(samples[FEATURES].describe().T[["count", "mean", "std", "min", "max"]].round(2))

X: 308,112 rows (917 run days x 336 hours), 12 features; rows with a known target: 303,211


,count,mean,std,min,max
nwp_lagged_anomaly,292264.0,-0.37,4.19,-21.00,15.50
nwp_anomaly,292264.0,-0.40,4.59,-25.25,16.68
nwp_recent_error,308112.0,1.10,1.54,-6.60,6.47
nwp_bias_14d,308112.0,0.81,0.69,-0.96,2.61
nwp_bias_30d,308112.0,0.81,0.61,-0.50,2.25
nwp_bias_60d,308112.0,0.81,0.57,-0.28,1.88
anom_now,308112.0,0.51,4.57,-13.54,15.46
anom_24h,308112.0,0.50,4.22,-13.46,13.69
hour_sin1,308112.0,0.00,0.71,-1.00,1.00
hour_cos1,308112.0,0.00,0.71,-1.00,1.00


## 5. Checks and export

In [6]:
final = samples[samples["run_day"].eq(pd.Timestamp(LAST_RUN_DAY))]
checks = {
    "every target is after its 11pm origin": (np.repeat(origins, HORIZON) < samples["target_utc"]).all(),
    "final forecast covers 12am Sep 17 - 11pm Sep 30": final["target_utc"].min() == CUTOFF and len(final) == HORIZON,
    "final forecast targets are unknown": final["temperature"].isna().all(),
    "final forecast has every feature": final[FEATURES].notna().all().all(),
    "every ECMWF run used was issued before its origin": (np.repeat(runs, HORIZON) < np.repeat(origins, HORIZON)).all(),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values())

SAMPLES_FILE.parent.mkdir(parents=True, exist_ok=True)
for path in (SAMPLES_FILE, SAMPLES_VIEW_FILE):
    samples.to_csv(path, index=False, float_format="%.8g")
    print(f"Saved {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024 ** 2:.1f} MB)")
display(final[["target_local", "lead_day", "hist_avg", "ecmwf_12z", "ecmwf_lagged_mean", *FEATURES[:3]]].iloc[[0, 1, 207, 335]].round(2))

,passed
every target is after its 11pm origin,True
final forecast covers 12am Sep 17 - 11pm Sep 30,True
final forecast targets are unknown,True
final forecast has every feature,True
every ECMWF run used was issued before its origin,True


Saved data/processed/model_samples_2024-03-14_to_2026-09-16.csv.gz (9.0 MB)
Saved data/processed/model_samples_2024-03-14_to_2026-09-16.csv (61.8 MB)


,target_local,lead_day,hist_avg,ecmwf_12z,ecmwf_lagged_mean,nwp_lagged_anomaly,nwp_anomaly,nwp_recent_error
307776,2026-09-17 00:00,1,20.02,19.9,20.07,0.05,-0.12,1.4
307777,2026-09-17 01:00,1,19.61,19.6,19.70,0.09,-0.01,1.4
307983,2026-09-25 15:00,9,26.38,23.7,25.47,-0.91,-2.68,1.4
308111,2026-09-30 23:00,14,19.14,17.4,17.40,-1.74,-1.74,1.4
